# AM Training

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/enph-479-edge-ai-stt/enph-479-edge-ai-stt/blob/main/training/notebooks/am_training.ipynb)

Trains the acoustic model (3x256 unidirectional LSTM + CTC) on LibriSpeech train-clean-100 and prints dev-clean loss and greedy CER after every epoch, then fine-tunes it with its weights quantized to 6 bits. **Runtime: T4 GPU.** Runs start to finish in one session, then saves both models, their training logs and the 6-bit model's weight image for the FPGA to a run folder on Google Drive.

Features are computed on the fly in the DataLoader workers. If GPU utilization in Colab's Resources panel stays low while the CPU is pegged, feature extraction is the bottleneck.

Specs (30-symbol vocab, feature constants) are provisional until `shared/specs` is frozen.

## 1. Pull the repo

In [ ]:
import importlib
import os
import subprocess
import sys

REPO_URL = "https://github.com/enph-479-edge-ai-stt/enph-479-edge-ai-stt.git"
REPO_DIR = "/content/enph-479-edge-ai-stt"
BRANCH = "feat/am-lr-schedule"  # set to a feature branch to test it before merging

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
# Always land on the tip of BRANCH, even if an earlier run left the clone on another branch.
git = ["git", "-C", REPO_DIR]
subprocess.run([*git, "fetch", "origin", BRANCH], check=True)
subprocess.run([*git, "checkout", "-B", BRANCH, f"origin/{BRANCH}"], check=True)
print(subprocess.run([*git, "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

# Installs this checkout's package and its dependencies (jiwer, soundfile);
# torch and torchaudio come with Colab.
training_dir = os.path.join(REPO_DIR, "training")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", training_dir], check=True)

# The editable install only takes effect at interpreter start, so put src/ on the
# path for this already-running kernel.
sys.path.insert(0, os.path.join(training_dir, "src"))
importlib.invalidate_caches()

## 2. Mount Drive and configure the run

Mounting Drive here means the permission prompt comes up now, not after training. The results go to `MyDrive/enph479-stt/runs/<start time>/` at the end.

One epoch over train-clean-100 takes about 9 minutes on a T4, so 40 epochs is about 6 hours. The 6-bit fine-tune adds 5 more epochs.

In [ ]:
import time

from google.colab import drive

from training.am.train import TrainConfig, load_data, train

drive.mount("/content/drive")
RUN_DIR = f"/content/drive/MyDrive/enph479-stt/runs/{time.strftime('%Y%m%d-%H%M')}"

cfg = TrainConfig(epochs=40)
print(RUN_DIR)
cfg

## 3. Data

Downloads train-clean-100 (6.3 GB) and dev-clean (337 MB) to `/content/data`, then computes CMVN from the training utterances. `cmvn` is that mean and std, saved with the weights at the end.

In [ ]:
datasets, cmvn = load_data("/content/data", cfg)

## 4. Train

Expect the blank wall first: test CER sits near 1.0 while the model outputs only blanks, then characters appear and it falls. Progress lines show ms/batch; each epoch line shows train and eval time.

The learning rate falls from `cfg.lr` to 0 on one cosine over the run. Each epoch line also shows the test loss next to the train loss: the model is overfitting once the test loss stops falling while the train loss keeps going. The train loss is measured with dropout on, so early in the run it can sit above the test loss.

In [ ]:
model = train(cfg, datasets, "train.log")

## 5. Fine-tune with 6-bit weights

Starts from the float model above and trains 5 more epochs, starting at a tenth of the learning rate, with every weight matrix quantized to 6 bits (63 levels) on the FPGA's fixed-point grid, the paper's retraining step. The CER printed here is the 6-bit model's. Activations and the cell state are still float.

In [ ]:
from dataclasses import replace

qcfg = replace(cfg, epochs=5, lr=3e-5, quantize=True)
qmodel = train(qcfg, datasets, "train_6bit.log", model.state_dict())

## 6. Save to Drive

Writes `am.pt` and `am_6bit.pt` (each the weights plus the CMVN mean/std the model's inputs were normalized with; the weights are useless without them) and both training logs to the run folder. Also writes the 6-bit model packed for the FPGA: `am_fabric.mem` is the contents of the fabric's weight memory in the order the board streams it in, and `am_fabric.json` is what the ARM needs to prepare the inputs (layout in `shared/specs/weight_image.md`). Then flushes Drive so the upload finishes before the session ends.

In [ ]:
from training.lstm.export import save_run

save_run(RUN_DIR, "am", model, qmodel, **cmvn)
print(f"saved both models, both training logs and the FPGA weight image to {RUN_DIR}")
drive.flush_and_unmount()